In [3]:
!pip install sqlalchemy

In [1]:
import pandas as pd
from sqlalchemy import create_engine

In [2]:
from google.colab import files
uploaded = files.upload()

Saving cast_cleaned.csv to cast_cleaned.csv
Saving crew_cleaned.csv to crew_cleaned.csv
Saving genres_cleaned.csv to genres_cleaned.csv
Saving movie_genres_cleaned.csv to movie_genres_cleaned.csv
Saving movie_keywords_cleaned.csv to movie_keywords_cleaned.csv
Saving movies_cleaned.csv to movies_cleaned.csv


In [6]:
#Creating databse
engine = create_engine('sqlite:///movies.db')

In [7]:
#reading the cleaned file
movies = pd.read_csv('movies_cleaned.csv')
genres = pd.read_csv('genres_cleaned.csv')
movie_genres = pd.read_csv('movie_genres_cleaned.csv')
cast = pd.read_csv('cast_cleaned.csv')
crew = pd.read_csv('crew_cleaned.csv')
movie_keywords = pd.read_csv('movie_keywords_cleaned.csv')

In [9]:
#Pushing dataset into database
movies.to_sql('movies', engine, if_exists='replace', index=False)
genres.to_sql('genres', engine, if_exists='replace', index=False)
movie_genres.to_sql('movie_genres', engine, if_exists='replace', index=False)
cast.to_sql('cast', engine, if_exists='replace', index=False)
crew.to_sql('crew', engine, if_exists='replace', index=False)
movie_keywords.to_sql('movie_keywords', engine, if_exists='replace', index=False)

41681

In [ ]:
#Checking tables are connected

query = """

   SELECT m.movie_id, m.title, g.genre_name
   FROM movies m
   JOIN movie_genres mg ON m.movie_id = mg.movie_id
   JOIN genres g ON mg.genre_id = g.genre_id
   LIMIT 10;

"""
result = pd.read_sql(query,engine)
result

In [ ]:
# 1.List all the action movies released after 2015

query = """
SELECT m.title, m.release_date
FROM movies AS m
INNER JOIN movie_genres AS mg ON m.movie_id = mg.movie_id
INNER JOIN genres AS g ON mg.genre_id = g.genre_id
WHERE g.genre_name = 'Action' AND m.release_date > '2015-12-31'
ORDER BY m.release_date;
"""
result = pd.read_sql(query, engine)
result

In [ ]:
# 2.Find the top 10 highest-grossing movies along with genres

query = """
SELECT m.title, m.revenue,
GROUP_CONCAT(g.genre_name) AS genre_name
FROM movies AS m
JOIN movie_genres AS mg ON m.movie_id = mg.movie_id
JOIN genres AS g ON mg.genre_id = g.genre_id
GROUP BY m.movie_id, m.title, m.revenue
ORDER BY m.revenue DESC
LIMIT 10;
"""
result = pd.read_sql(query, engine)
result

In [ ]:
# 3.Calculate the average budget and average revenue for each genre

query = """
SELECT g.genre_name, AVG(m.budget) AS average_budget, AVG(m.revenue) AS average_revenue
FROM movies AS m
JOIN movie_genres AS mg ON m.movie_id = mg.movie_id
JOIN genres AS g ON mg.genre_id = g.genre_id
GROUP BY g.genre_name ORDER BY average_revenue DESC;
"""
result = pd.read_sql(query, engine)
result

In [ ]:
# 4.List the top 10 actors who have appeared in most movies

query = """
SELECT actor_name, COUNT(DISTINCT movie_id) AS total_movies
FROM cast
GROUP BY actor_name ORDER BY total_movies DESC
LIMIT 10;
"""
result = pd.read_sql(query, engine)
result

In [ ]:
# 5.Find all the director who have directed more than 3 moveies along with movie count

query = """
SELECT person_name AS director_name, COUNT(DISTINCT movie_id) AS movie_count
FROM crew
WHERE job = 'Director'
GROUP BY person_id, person_name
HAVING COUNT(DISTINCT movie_id) > 3 ORDER BY movie_count DESC;
"""
result = pd.read_sql(query, engine)
result

In [ ]:
# 6.List the top 10 most frequently used keyword across all movies

query = """
SELECT keyword_name, COUNT(DISTINCT movie_id) AS movie_count
FROM movie_keywords
GROUP BY keyword_id, keyword_name ORDER BY movie_count DESC
LIMIT 10;
"""
result = pd.read_sql(query, engine)
result

In [ ]:
# 7.Find movies where the budget was above overall average budget but revenue was below the overall average revenue

query = """
SELECT title, budget, revenue
FROM movies
WHERE budget > (SELECT AVG(budget) FROM movies) AND revenue < (SELECT AVG(revenue) FROM movies)
ORDER BY budget DESC;
"""
result = pd.read_sql(query, engine)
result

In [ ]:
# 8. List all actors who have appeared in a movie directed by a specific director

query = """
SELECT DISTINCT c.actor_name
FROM "cast" c
JOIN crew cr ON c.movie_id = cr.movie_id
WHERE cr.job = 'Director' AND cr.person_name = 'Quentin Tarantino';
"""
result = pd.read_sql(query, engine)
result

In [ ]:
# 9.Find the genre with highest average rating, only genre with atleast 20 movies

query = """
SELECT g.genre_name, AVG(m.vote_average) AS average_rating
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
GROUP BY g.genre_name
HAVING COUNT(DISTINCT m.movie_id) >= 20
ORDER BY average_rating DESC
LIMIT 1;
"""
result = pd.read_sql(query, engine)
result

In [ ]:
# 10.List the top 10 movies with the highest number of keywords tagged to them

query = """
SELECT m.title, COUNT(mk.keyword_id) AS keyword_count
FROM movies m
JOIN movie_keywords mk ON m.movie_id = mk.movie_id
GROUP BY m.movie_id, m.title
ORDER BY keyword_count DESC
LIMIT 10;
"""
result = pd.read_sql(query, engine)
result